<a href="https://colab.research.google.com/github/Tomax47/Uni-Crawler-Tasks/blob/main/crawler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import zipfile
from google.colab import files
import time

In [4]:
SEED_URL = "https://ru.wikipedia.org/wiki/%D0%98%D0%BD%D1%84%D0%BE%D1%80%D0%BC%D0%B0%D1%86%D0%B8%D0%BE%D0%BD%D0%BD%D1%8B%D0%B9_%D0%BF%D0%BE%D0%B8%D1%81%D0%BA"
MAX_PAGES = 100
OUTPUT_DIR = "pages"
INDEX_FILE = "index.txt"
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

In [5]:
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

def is_valid_url(url, base_domain):
    parsed = urlparse(url)
    return bool(parsed.netloc) and parsed.netloc == base_domain

def crawl(seed_url, max_pages):
    visited = set()
    queue = [seed_url]
    count = 0
    base_domain = urlparse(seed_url).netloc

    with open(INDEX_FILE, "w", encoding="utf-8") as index_f:
        while queue and count < max_pages:
            url = queue.pop(0)

            if url in visited or "#" in url:
                continue

            try:
                response = requests.get(url, headers=HEADERS, timeout=10)

                if response.status_code != 200:
                    print(f"Skipping {url} (Status: {response.status_code})")
                    continue

                content_type = response.headers.get('Content-Type', '')
                if 'text/html' not in content_type:
                    continue

                file_name = f"{count}.html"
                file_path = os.path.join(OUTPUT_DIR, file_name)

                with open(file_path, "w", encoding="utf-8") as f:
                    f.write(response.text)

                index_f.write(f"{count} {url}\n")
                visited.add(url)
                print(f"[{count}] Saved: {url}")
                count += 1

                soup = BeautifulSoup(response.text, 'html.parser')
                for link in soup.find_all('a', href=True):
                    full_url = urljoin(url, link['href'])
                    full_url = full_url.split('?')[0].split('#')[0]

                    if is_valid_url(full_url, base_domain) and full_url not in visited:
                        queue.append(full_url)

                time.sleep(0.1)

            except Exception as e:
                print(f"Error crawling {url}: {e}")

    return count

In [6]:
print(f"Starting crawl at: {SEED_URL}")
total_downloaded = crawl(SEED_URL, MAX_PAGES)
print(f"\nFinished. Total pages downloaded: {total_downloaded}")

if total_downloaded > 0:
    def create_archive(zip_name, folder_path, index_path):
        with zipfile.ZipFile(zip_name, 'w') as zipf:
            zipf.write(index_path, arcname=index_path)
            for root, dirs, files_list in os.walk(folder_path):
                for file in files_list:
                    zipf.write(os.path.join(root, file),
                               arcname=os.path.join(folder_path, file))

    archive_name = "tema1_result.zip"
    create_archive(archive_name, OUTPUT_DIR, INDEX_FILE)
    print(f"Downloading {archive_name}...")
    files.download(archive_name)
else:
    print("No pages were downloaded")

Starting crawl at: https://ru.wikipedia.org/wiki/%D0%98%D0%BD%D1%84%D0%BE%D1%80%D0%BC%D0%B0%D1%86%D0%B8%D0%BE%D0%BD%D0%BD%D1%8B%D0%B9_%D0%BF%D0%BE%D0%B8%D1%81%D0%BA
[0] Saved: https://ru.wikipedia.org/wiki/%D0%98%D0%BD%D1%84%D0%BE%D1%80%D0%BC%D0%B0%D1%86%D0%B8%D0%BE%D0%BD%D0%BD%D1%8B%D0%B9_%D0%BF%D0%BE%D0%B8%D1%81%D0%BA
[1] Saved: https://ru.wikipedia.org/wiki/%D0%90%D0%BD%D0%B3%D0%BB%D0%B8%D0%B9%D1%81%D0%BA%D0%B8%D0%B9_%D1%8F%D0%B7%D1%8B%D0%BA
[2] Saved: https://ru.wikipedia.org/wiki/%D0%9F%D0%BE%D0%B8%D1%81%D0%BA
[3] Saved: https://ru.wikipedia.org/wiki/%D0%98%D0%BD%D1%84%D0%BE%D1%80%D0%BC%D0%B0%D1%86%D0%B8%D1%8F
[4] Saved: https://ru.wikipedia.org/wiki/%D0%98%D0%BD%D1%84%D0%BE%D1%80%D0%BC%D0%B0%D1%86%D0%B8%D0%BE%D0%BD%D0%BD%D1%8B%D0%B5_%D0%BF%D0%BE%D1%82%D1%80%D0%B5%D0%B1%D0%BD%D0%BE%D1%81%D1%82%D0%B8
[5] Saved: https://ru.wikipedia.org/wiki/%D0%9D%D0%B0%D1%83%D0%BA%D0%B0
[6] Saved: https://ru.wikipedia.org/w/index.php
[7] Saved: https://ru.wikipedia.org/wiki/1948
[8] Saved: https:/

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Task-2: Tokenizaiton

In [7]:
!pip install pymorphy3 nltk

import re
import nltk
from pymorphy3 import MorphAnalyzer
from nltk.corpus import stopwords

nltk.download('stopwords')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 52.0 MB/s eta 0:00:00


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [8]:
INPUT_DIR = "pages"
TOKENS_FILE = "tokens.txt"
LEMMAS_FILE = "lemmas.txt"

morph = MorphAnalyzer()
russian_stopwords = set(stopwords.words('russian'))
EXCLUDED_POS = {'PREP', 'CONJ', 'PRCL', 'INTJ'}

In [9]:
def is_valid_word(word):
    return bool(re.fullmatch(r'[а-яё]+', word))

def extract_and_process(directory):
    unique_tokens = set()
    lemma_map = {}

    if not os.path.exists(directory):
        print(f"Error: {directory} folder not found")
        return unique_tokens, lemma_map

    for filename in sorted(os.listdir(directory)):
        if filename.endswith(".html"):
            file_path = os.path.join(directory, filename)
            with open(file_path, 'r', encoding='utf-8') as f:
                soup = BeautifulSoup(f.read(), 'html.parser')

                for element in soup(["script", "style", "meta", "noscript"]):
                    element.decompose()

                text = soup.get_text().lower()
                potential_words = re.findall(r'[а-яё0-9a-z]+', text)

                for word in potential_words:
                    if not is_valid_word(word) or len(word) < 2:
                        continue

                    if word in russian_stopwords:
                        continue

                    analysis = morph.parse(word)[0]
                    pos = analysis.tag.POS
                    if pos in EXCLUDED_POS:
                        continue

                    unique_tokens.add(word)

                    lemma = analysis.normal_form
                    if lemma not in lemma_map:
                        lemma_map[lemma] = set()
                    lemma_map[lemma].add(word)

    return sorted(list(unique_tokens)), lemma_map

In [10]:
print("Processing...")
tokens, lemmas = extract_and_process(INPUT_DIR)

with open(TOKENS_FILE, "w", encoding="utf-8") as f:
    for token in tokens:
        f.write(f"{token}\n")

with open(LEMMAS_FILE, "w", encoding="utf-8") as f:
    for lemma, token_set in sorted(lemmas.items()):
        token_string = " ".join(sorted(list(token_set)))
        f.write(f"{lemma}: {token_string}\n")

print(f"DOne!")

files.download(TOKENS_FILE)
files.download(LEMMAS_FILE)

Processing documents... this may take a moment.
Extraction complete!
Unique tokens found: 28629
Unique lemmas found: 14440


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>